# Badge 1: Setup (run once, as ACCOUNTADMIN)

Creates the isolated environment the lesson notebooks run in:

| Course object | Isolated stand-in | Notes |
|---|---|---|
| ACCOUNTADMIN role | `BADGE1_DWW_ADMIN` | holds the account privileges the course uses as ACCOUNTADMIN |
| SYSADMIN role | `BADGE1_DWW_ROLE` | granted to `BADGE1_DWW_ADMIN`, mirroring SYSADMIN under ACCOUNTADMIN |
| COMPUTE_WH | `BADGE1_DWW_WH` | X-Small, owned by `BADGE1_DWW_ADMIN` (the Lesson 2 challenge transfers it) |
| UTIL_DB | `UTIL_DB` | shared across badges; owned by SYSADMIN; the course assumes it exists (see audit note) |
| learner's Downloads folder | `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` | holds the course data files the learner would download |

Hierarchy: `BADGE1_DWW_ROLE` → `BADGE1_DWW_ADMIN` → SYSADMIN → ACCOUNTADMIN, so account admins still see everything.
Object names (DEMO_DB, GARDEN_PLANTS, …) are the course's exact names.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH

## Check for name collisions
The course's database names are account-wide. This lists any that already exist.

In [ ]:
use role accountadmin;
show databases;
select "name", "owner" from table(result_scan(last_query_id()))
where "name" in ('DEMO_DB','GARDEN_PLANTS','UTIL_DB','LIBRARY_CARD_CATALOG','SOCIAL_MEDIA_FLOODGATES');

## Roles and grants

In [ ]:
use role accountadmin;
create role if not exists {{ADMIN_ROLE}} comment = 'Badge 1 stand-in for ACCOUNTADMIN';
create role if not exists {{SYSADMIN_ROLE}} comment = 'Badge 1 stand-in for SYSADMIN';
grant role {{SYSADMIN_ROLE}} to role {{ADMIN_ROLE}};
grant role {{ADMIN_ROLE}} to role sysadmin;
set badge_user = current_user();
grant role {{ADMIN_ROLE}} to user identifier($badge_user);
grant role {{SYSADMIN_ROLE}} to user identifier($badge_user);
-- trial ACCOUNTADMIN and SYSADMIN can both create databases
grant create database on account to role {{ADMIN_ROLE}};
grant create database on account to role {{SYSADMIN_ROLE}};

## Warehouse (stands in for COMPUTE_WH, owned by the ACCOUNTADMIN stand-in like in a trial)

In [ ]:
use role accountadmin;
create warehouse if not exists {{WAREHOUSE}}
  warehouse_size = xsmall auto_suspend = 60 auto_resume = true initially_suspended = true
  comment = 'Badge 1 stand-in for COMPUTE_WH';
grant ownership on warehouse {{WAREHOUSE}} to role {{ADMIN_ROLE}} copy current grants;

## UTIL_DB (shared across badges)

> ⚠️ **Audit note:** The course never tells learners to create UTIL_DB, yet Lesson 2's checklist expects SYSADMIN to see it, and the DORA GRADER (Lesson 6) and MY_INTERNAL_STAGE (Lesson 9) live in it. It is probably provisioned by the course's special trial sign-up link; that could not be verified. Other badges use UTIL_DB too, so it is owned by SYSADMIN and each badge's roles get only the privileges they need.

In [ ]:
use role accountadmin;
create database if not exists UTIL_DB comment = 'Shared utility database for badge course audits (DORA grader, course stages)';
grant ownership on database UTIL_DB to role sysadmin copy current grants;
grant ownership on schema UTIL_DB.PUBLIC to role sysadmin copy current grants;
grant usage on database UTIL_DB to role {{ADMIN_ROLE}};
grant usage on database UTIL_DB to role {{SYSADMIN_ROLE}};
grant usage, create stage, create file format, create workspace on schema UTIL_DB.PUBLIC to role {{ADMIN_ROLE}};
grant usage on schema UTIL_DB.PUBLIC to role {{SYSADMIN_ROLE}};

## Course data files (the learner's Downloads folder)
The course has learners download 8 files and upload them through Snowsight. Here they sit on one stage, and each lesson copies them to the stage the course uses. The files are in `data/` beside these notebooks; upload them once with the Snowflake CLI:

```
snow sql -q "use role BADGE1_DWW_ADMIN; put 'file://<path>/data/*' @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS auto_compress=false overwrite=true"
```

or in Snowsight: Catalog → UTIL_DB → PUBLIC → Stages → BADGE1_DOWNLOADS → **+ Files**.

In [ ]:
use role {{ADMIN_ROLE}};
create stage if not exists UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
  comment = 'Badge 1: stands in for the learner''s local Downloads folder (course data files)'
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');
grant read on stage UTIL_DB.PUBLIC.BADGE1_DOWNLOADS to role {{SYSADMIN_ROLE}};
alter stage UTIL_DB.PUBLIC.BADGE1_DOWNLOADS refresh;
list @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS;

In [ ]:
expected = ['LU_SOIL_TYPE.tsv', 'VEG_NAME_TO_SOIL_TYPE_PIPE.txt', 'author_with_header.json', 'json_book_author_nested.txt', 'nutrition_tweets.json', 'veg_plant_height.csv', 'veggie_details_a_to_k_comm.csv', 'veggie_details_k_to_z_pipe.csv']
found = sorted(n.split('/')[-1] for n in create_downloads_stage['name'])
print('Missing course files:', sorted(set(expected) - set(found)) or 'none')

## Keep each role's view isolated
With secondary roles on, Snowsight shows the combined privileges of all your roles. Each lesson notebook runs `use secondary roles none`. To make Snowsight isolated too, uncomment the statement below.

In [ ]:
-- alter user identifier($badge_user) set default_secondary_roles = ();
select current_user(), current_secondary_roles();

## Teardown (commented out)
Removes everything Badge 1 creates. Uncomment to reset.

In [ ]:
-- use role accountadmin;
-- drop database if exists DEMO_DB;
-- drop database if exists GARDEN_PLANTS;
-- drop database if exists LIBRARY_CARD_CATALOG;
-- drop database if exists SOCIAL_MEDIA_FLOODGATES;
-- UTIL_DB is shared with other badges; drop only Badge 1's objects in it
-- drop stage if exists UTIL_DB.PUBLIC.MY_INTERNAL_STAGE;
-- drop stage if exists UTIL_DB.PUBLIC.BADGE1_DOWNLOADS;
-- drop resource monitor if exists daily_3_credit_limit;
-- drop warehouse if exists {{WAREHOUSE}};
-- drop role if exists {{SYSADMIN_ROLE}};
-- drop role if exists {{ADMIN_ROLE}};
select 'teardown is commented out' as status;